## Testing hmbert-ajmc via `flair`

In [5]:
from flair.data import Sentence, Token
from flair.models import SequenceTagger

/Users/matteo/.pyenv/versions/3.10.0/envs/ajmc-inception-recommender/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
tagger = SequenceTagger.load("hmteams/flair-hipe-2022-ajmc-fr")

/Users/matteo/.pyenv/versions/3.10.0/envs/ajmc-inception-recommender/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2023-11-01 15:45:58,693 SequenceTagger predicts: Dictionary with 25 tags: O, S-scope, B-scope, E-scope, I-scope, S-pers, B-pers, E-pers, I-pers, S-work, B-work, E-work, I-work, S-loc, B-loc, E-loc, I-loc, S-object, B-object, E-object, I-object, S-date, B-date, E-date, I-date


In [12]:
text = "— 469 . Πεδία . Les tribraques formés par un seul mot sont rares chez les tragiques et Homère, partont ailleurs qu ’ au premier pied . CÉ . cependant QEd , Roi , 719 , 826 , 4496 ."
sen = Sentence(text)
tagger.predict(sen)

In [13]:
print(sen)

Sentence[41]: "— 469 . Πεδία . Les tribraques formés par un seul mot sont rares chez les tragiques et Homère, partont ailleurs qu ’ au premier pied . CÉ . cependant QEd , Roi , 719 , 826 , 4496 ." → ["Homère"/pers, "QEd , Roi"/work, "719"/scope, "826"/scope, "4496"/scope]


In [14]:
sen.get_spans('ner')

[Span[18:19]: "Homère" → pers (0.9998),
 Span[31:34]: "QEd , Roi" → work (0.9992),
 Span[35:36]: "719" → scope (0.9999),
 Span[37:38]: "826" → scope (0.9999),
 Span[39:40]: "4496" → scope (0.9998)]

## Testing our basic custom classifier

In [4]:
import sys
sys.path.append('.')
from inception_recommender import *

/Users/matteo/.pyenv/versions/3.10.0/envs/ajmc-inception-recommender/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
85288-4593210880 2023-11-02 12:26:16,147 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): huggingface.co:443
85288-4593210880 2023-11-02 12:26:16,652 - urllib3.connectionpool - DEBUG - https://huggingface.co:443 "HEAD /hmteams/flair-hipe-2022-ajmc-fr/resolve/main/pytorch_model.bin HTTP/1.1" 302 0


2023-11-02 12:26:24,840 SequenceTagger predicts: Dictionary with 25 tags: O, S-scope, B-scope, E-scope, I-scope, S-pers, B-pers, E-pers, I-pers, S-work, B-work, E-work, I-work, S-loc, B-loc, E-loc, I-loc, S-object, B-object, E-object, I-object, S-date, B-date, E-date, I-date


In [25]:
ajmc_fr_classif = ClassicsNERClassifier("fr", "coarse")

2023-11-02 08:32:06,408 SequenceTagger predicts: Dictionary with 25 tags: O, S-scope, B-scope, E-scope, I-scope, S-pers, B-pers, E-pers, I-pers, S-work, B-work, E-work, I-work, S-loc, B-loc, E-loc, I-loc, S-object, B-object, E-object, I-object, S-date, B-date, E-date, I-date


In [26]:
cas_doc = load_test_document()
ajmc_fr_classif.predict(cas_doc, PREDICTED_TYPE, PREDICTED_FEATURE, PROJECT_ID, '01', USER)

In [27]:
for prediction in cas_doc.select(PREDICTED_TYPE):
    print(prediction.value, prediction.get_covered_text())

pers Hérodute,
pers ΠῚ,
work ταχνιπς
scope IX, Lxxi1.
pers Euripide,
work Médée,
scope 400;
work Hécube,
scope 4044
work Hercule furieux,
scope 1400.
pers οἶνον,
work γεύεσθαί


## Testing the recommender server

In [5]:
import json
from typing import Any
import requests

def _send_json(url: str, body: Any):

    response = requests.post(
        url,
        data=json.dumps(body).encode("utf-8"),
        headers={"content-type": "application/json"},
        verify=False
    )
    return response


def prepare_test_request(user_id, project_id, document_id):

    with open("data/AjMC_TypeSystem.xml", "rb") as f:
        typesystem = merge_typesystems(load_typesystem(f), build_typesystem())

    with open("data/lestragdiesdeso00tourgoog_0065.xmi", "rb") as f:
        cas = load_cas_from_xmi(f, typesystem=typesystem)

    request = {
        "typeSystem": typesystem.to_xml(),
        "document": {
            "xmi": cas.to_xmi(),
            "documentId": document_id,
            "userId": user_id,
        },
        "metadata": {
            "layer": PREDICTED_TYPE,
            "feature": PREDICTED_FEATURE,
            "projectId": project_id,

        }
    }
    return request

In [6]:
recommender_predict_url = "http://127.0.0.1:5000/ajmc_ner_fr/predict"
req = prepare_test_request(user_id="admin", project_id="1", document_id=1)
response = _send_json(recommender_predict_url, req)

85288-4593210880 2023-11-02 12:26:31,356 - urllib3.connectionpool - DEBUG - Starting new HTTP connection (1): 127.0.0.1:5000
85288-4593210880 2023-11-02 12:26:32,368 - urllib3.connectionpool - DEBUG - http://127.0.0.1:5000 "POST /ajmc_ner_fr/predict HTTP/1.1" 200 39034


In [120]:
body = response.json()

In [121]:
typesystem = load_typesystem(req["typeSystem"])
cas = load_cas_from_xmi(body["document"], typesystem)
layer = req["metadata"]["layer"]
feature = req["metadata"]["feature"]

In [122]:
for pred in cas.select(layer):
    print(pred.get_covered_text(), pred.value)

Hérodute, pers
ΠῚ, pers
ταχνιπς work
IX, Lxxi1. scope
Euripide, pers
Médée, work
400; scope
Hécube, work
4044 scope
Hercule furieux, work
1400. scope
οἶνον, pers
γεύεσθαί work
